## Read in contextual ERA5 variables pre-onset, during onset and post seasonal onset

From ERA5 Land starting with:
- Surface net short-wave radiation
- 2 metre temperature

From ERA5 surface starting with:
- Low cloud cover
- Medium cloud cover
- High cloud cover

From ERA5 pressure levels starting with:
- u, v, q --> mois flux convergence
- q, pressure, temperature --> dewpoint temperature --> equivalent potential temperature --> conditional instability
    - using MetPy for this

Might want to include:
- column water vapour

Will want to compare this with:
- MODIS evaporation
- MODIS FAPAR

In [ ]:
import sys
import xarray as xr
import numpy as np
from matplotlib import pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature
import geopandas as gpd
from shapely.geometry import mapping
import pandas as pd
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

In [ ]:
datap = "/Users/ellendyer/Documents/GitHub/Isotopes_F4R/plots/"
datael = "/Volumes/ESA_F4R/era/era5/pressure_levels/"
dataes = "/Volumes/ESA_F4R/era/era5/surface/"
dataep = "/Volumes/ESA_F4R/era/era5/era5_surface/"
band = {'N':[5,12,10,31],'EQ':[-5,5,8,29],'S':[-15,-5,12,31]}
Y1=1990
Y2=2024
B = 'EQ'
lon_bnds, lat_bnds = (band[B][2], band[B][3]), (band[B][0],band[B][1])
time_bnds = (str(Y1)+'-01-01',str(Y2)+'-12-31')



### Calculate equivalent potential temperature

In [ ]:
#https://unidata.github.io/MetPy/latest/api/generated/metpy.calc.dewpoint_from_specific_humidity.html
#https://unidata.github.io/MetPy/latest/api/generated/metpy.calc.equivalent_potential_temperature.html
from metpy.calc import equivalent_potential_temperature
from metpy.calc import dewpoint_from_specific_humidity
from metpy.units import units

#Reading in pressure level variables from ERA5
ds_era_pres_avg = xr.open_dataset("/Volumes/ESA_F4R/ed_prepare/analysis/ds_era_pres_avg.nc")
plev = ds_era_pres_avg['plev']/100.0  
plev = plev.attrs['units'] = 'hPa'
q = 1000.0*ds_era_pres_avg['q']
q = q.attrs['units'] = 'g/kg'
t = ds_era_pres_avg['t']-273.15 
t = t.attrs['units'] = 'degC'

dewpoint = dewpoint_from_specific_humidity(plev,q) #pressure * units.hPa, specific humidity * units('g/kg')
theta = equivalent_potential_temperature(plev,t,dewpoint) #pressure *units.hPa, temperature *units.degC, dewpoint*units.degC


### Calculate moisture flux convergence

In [ ]:
#https://geocat-comp.readthedocs.io/en/v2023.06.0/examples/vimfc.html#calculate-moisture-flux-convergence-mfc
import geocat.comp as gc

ds_era_pres = xr.open_dataset("/Volumes/ESA_F4R/ed_prepare/analysis/ds_era_pres.nc")
plev = ds_era_pres['plev']/100.0  
q = 1000.0*ds_era_pres['q']
t = ds_era_pres['t']-273.15 
sp = xr.open_dataset("/Volumes/ESA_F4R/ed_prepare/analysis/ds_era_psfc.nc")

delta_pressure_levels = gc.meteorology.delta_pressure(pressure_lev=plev, surface_pressure=sp)
g = 9.80665 # gravitational acceleration (m s-2)
layer_mass_weighting = delta_pressure_levels / g # Layer Mass Weighting
layer_mass_weighting .attrs["units"] = "kg m-2"

mass_weighted_vapor = q * layer_mass_weighting # mass weighted 'q'
iq = mass_weighted_vapor.sum(dim="plev") # Vertically Integrated Vapor
iq.attrs["units"] = "g m-2"

du_dx, du_dy = gc.gradient(ds_era_pres.u) # (s-1)
dv_dx, dv_dy = gc.gradient(ds_era_pres.v)
dq_dx, dq_dy = gc.gradient(iq) # (g m-3)

advection = (-ds_era_pres.u * dq_dx) - (ds_era_pres.v * dq_dy) # (g s-1 m-2)
convergence = iq * np.add(du_dx, dv_dy) # (g s-1 m-2)

mfc = advection - convergence # moisture flux convergence (g m-2 s-1)